# Dia 08 — Bancos de dados

**🃏 Cartas:** 8 · 29 · 50 &nbsp;·&nbsp; **📘 Original:** *Day 8. Databases*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 8 — O dump do painel

Quinta-feira, 16h30. A planilha padronizada de ontem ainda está aberta quando a Delegada Rita entra
na sala do NFA com um envelope pardo e um pen drive lacrado.

— Um ex-funcionário da hospedagem que atendia a loja procurou a delegacia. Diz que isso é uma cópia
do banco de dados do painel administrativo. Veio com termo de entrega assinado.

A Iara confere o lacre, anota a hora e só então pluga o pen drive. Dentro, um único arquivo:
**`painel_dump.sql`**.

— Primeiro o hash — ela diz, antes que você pergunte. — Depois a gente abre. E abre em memória, sem
espalhar cópia pelo disco.

Você roda o SHA-256, anota no termo, e carrega o dump no SQLite. Duas tabelas: **pedidos** e
**contas_recebimento**. Uma coluna chamada `conta_id` liga uma à outra.

— Esse fio é tudo — diz a Iara, apontando para a tela. — Até agora a gente sabia **quem perdeu**.
Com um `JOIN`, a gente descobre **quem recebeu**. Três contas, três titulares. E olha o sobrenome da
terceira.

A Delegada se inclina sobre seu ombro. Pela primeira vez no caso, o dinheiro tem destino.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-08-bancos-de-dados")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Entender o que é um **dump `.sql`** e por que ele aparece em investigações.
- Converter um dump em CSV (a prática proposta pelo autor), sem enviar nada a sites de terceiros.
- Explorar um banco desconhecido: `sqlite_master`, `PRAGMA table_info`, `LIMIT`.
- Responder perguntas com `JOIN`, `GROUP BY`, `COUNT` e `SUM` (em centavos!).
- Criar o banco do caso e consultar com **parâmetros `?`**, entendendo o SQL injection.

## 🧠 Conceito em 2 minutos

Um **banco de dados relacional** guarda tabelas, como planilhas, que se ligam por colunas em comum.
Um **dump** é o banco inteiro "impresso" como texto: os comandos `CREATE TABLE` (a estrutura) e
`INSERT` (as linhas). Qualquer programa de banco consegue reconstruí-lo.

```sql
SELECT c.titular, COUNT(*), SUM(p.valor)      -- 3. o que mostrar
  FROM pedidos p                              -- 1. de onde
  JOIN contas_recebimento c ON c.id = p.conta_id  -- 2. ligando as tabelas
 GROUP BY c.id                                -- 4. um resultado por conta
```

O Python já vem com o **SQLite** (`import sqlite3`): um banco inteiro num único arquivo, sem servidor.
É perfeito para análise: você importa o que tem (CSV, dump) e faz perguntas em SQL.

Duas regras de ouro: **dinheiro em centavos inteiros** (o tipo `REAL` é float e erra), e **nunca
monte SQL colando texto** do usuário: use `?` e passe os valores separados.

## 🥋 Nível 1 — Exercício original (cipher387)

O Dia 8 original **não tem código**: o autor dá conselhos sobre dumps `.sql`, o Google Hacking
Database e conversores de SQL para CSV. Resumimos tudo em [`original/LEIA-ME.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/original/LEIA-ME.md)
e reconstruímos a prática proposta, só com biblioteca padrão:

| Arquivo | O que faz |
|---|---|
| [`original/LEIA-ME.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/original/LEIA-ME.md) | Resumo da proposta do autor, com nossas palavras, e o alerta sobre vazamentos reais |
| [`original/nivel1_sql_para_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/original/nivel1_sql_para_csv.py) | **Reconstrução** (não é código do autor): dump fictício → SQLite em memória → um CSV por tabela |

```bash
cd dia-08-bancos-de-dados
python original/nivel1_sql_para_csv.py
```

> 💡 O autor sugere conversores online de SQL para CSV. Para dados de caso, **não use**: você estaria
> enviando evidência (e dados pessoais) para o servidor de um desconhecido. O script local faz o mesmo.

### `original/nivel1_sql_para_csv.py` — Converter um dump .sql em arquivos CSV (uma tabela por arquivo)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Converter um dump .sql em arquivos CSV (uma tabela por arquivo)
# NÃO HÁ CÓDIGO DO AUTOR NESTE DIA. Esta é uma RECONSTRUÇÃO da prática proposta
# por cipher387 (Cyb_detective) no Dia 8 de "Python for OSINT. 21-day course":
# "converta o dump para CSV e abra numa planilha". Escrita para este repositório,
# só com a biblioteca padrão, usando o dump FICTÍCIO de caso/dados/.
# Licença: MIT (a mesma do repositório).
# ──────────────────────────────────────────────────────────────────────────

import csv
import sqlite3            # banco de dados SQLite: vem com o Python
from pathlib import Path

# Pasta do dia (funciona como script e no Colab)
PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
DUMP = PASTA_DIA.parent / "caso" / "dados" / "painel_dump.sql"
SAIDAS = PASTA_DIA / "saidas"

# 1. Um banco "de mentirinha" que vive só na memória RAM (some ao fim do script).
#    Assim nada do dump é gravado em disco sem querermos.
banco = sqlite3.connect(":memory:")

# 2. executescript roda todos os comandos do arquivo de uma vez (CREATE TABLE, INSERT…).
#    Isto funciona porque o dump fictício foi escrito em SQL compatível com SQLite.
#    Dumps de MySQL/PostgreSQL reais costumam ter comandos que o SQLite não entende.
banco.executescript(DUMP.read_text(encoding="utf-8"))

# 3. Descobrir quais tabelas existem (o SQLite guarda isso na tabela especial sqlite_master).
tabelas = [linha[0] for linha in banco.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")]

# 4. Exportar cada tabela para um CSV com cabeçalho.
SAIDAS.mkdir(exist_ok=True)
for tabela in tabelas:
    # Nome de tabela não pode ir como parâmetro "?"; aqui ele vem do próprio sqlite_master.
    cursor = banco.execute(f'SELECT * FROM "{tabela}"')
    colunas = [descricao[0] for descricao in cursor.description]   # nomes das colunas
    destino = SAIDAS / f"{tabela}.csv"
    with open(destino, "w", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.writer(arquivo, delimiter=";")
        escritor.writerow(colunas)
        escritor.writerows(cursor)                                  # todas as linhas
    total = banco.execute(f'SELECT COUNT(*) FROM "{tabela}"').fetchone()[0]
    print(f"💾 {tabela:<20} {total:>3} linhas → {destino.relative_to(PASTA_DIA)}")

banco.close()
print("\n✅ Pronto: abra os CSVs no Excel, LibreOffice ou Google Planilhas.")

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão** e 100% offline (os dados são os fictícios de `caso/dados/`).

| Arquivo | O que faz |
|---|---|
| [`br/01_explorar_dump.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/br/01_explorar_dump.py) | Hash do dump, tabelas (`sqlite_master`), colunas (`PRAGMA table_info`), contagens e primeiras linhas |
| [`br/02_quem_recebeu.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/br/02_quem_recebeu.py) | **O caso:** `JOIN` pedidos × contas, total por conta laranja em R$ e conferência com a planilha |
| [`br/03_banco_do_caso.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-08-bancos-de-dados/br/03_banco_do_caso.py) | Cria `saidas/caso.db` com as denúncias, consulta por mês e cidade e demonstra SQL injection |

```bash
cd dia-08-bancos-de-dados
python br/01_explorar_dump.py
python br/02_quem_recebeu.py
python br/03_banco_do_caso.py
```

**O que você deve ver no caso:**

| Conta (titular) | Pedidos | Recebido |
|---|---|---|
| Jéssica Prado Lima | 16 | R$ 21.242,10 |
| M. R. Brandão Comércio ME | 10 | R$ 17.840,90 |
| Wellington Arruda Paz | 11 | R$ 15.794,40 |
| **Total** | **37** | **R$ 54.877,40** |

O total do painel bate **centavo por centavo** com a planilha das vítimas. E a terceira conta tem o
mesmo sobrenome do vendedor "Kau": uma hipótese para investigar, não uma conclusão.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_explorar_dump.py` — Explorar um dump SQL desconhecido: que tabelas, que colunas, quantas linhas?

In [ ]:
%cd -q {PASTA_NB}
"""
Explorar um dump SQL desconhecido: que tabelas, que colunas, quantas linhas?

Carta 29 (rank B) — Jutsu do Arquivo Morto

O que você aprende:
  • a carregar um .sql no SQLite (biblioteca padrão) sem instalar servidor de banco;
  • a listar tabelas pela tabela especial sqlite_master;
  • a ver as colunas e os tipos com PRAGMA table_info;
  • a espiar as primeiras linhas com LIMIT antes de fazer qualquer conta.

Regra da Iara: "antes de perguntar ao banco, descubra o que o banco sabe".

Como rodar (dentro da pasta dia-08-bancos-de-dados):
    python br/01_explorar_dump.py
"""

import hashlib              # impressão digital do arquivo (cadeia de custódia, Dia 14)
import sqlite3
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
DUMP = PASTA_DIA.parent / "caso" / "dados" / "painel_dump.sql"

# ── 0. Antes de abrir: registrar o hash do arquivo recebido ───────────────
# Se alguém questionar "esse dump foi alterado?", o SHA-256 responde.
conteudo = DUMP.read_bytes()
print(f"🧾 Arquivo: {DUMP.name} · {len(conteudo)} bytes")
print(f"🔐 SHA-256: {hashlib.sha256(conteudo).hexdigest()}\n")

# ── 1. Carregar em memória ────────────────────────────────────────────────
banco = sqlite3.connect(":memory:")
banco.executescript(conteudo.decode("utf-8"))

# ── 2. Que tabelas existem? ───────────────────────────────────────────────
# sqlite_master é o "índice" do banco: guarda nome e o comando que criou cada tabela.
tabelas = banco.execute(
    "SELECT name, sql FROM sqlite_master WHERE type = 'table' ORDER BY name").fetchall()
print(f"🗄️  {len(tabelas)} tabela(s) encontrada(s):")

for nome, comando in tabelas:
    total = banco.execute(f'SELECT COUNT(*) FROM "{nome}"').fetchone()[0]
    print(f"\n── {nome} ({total} linhas) ──")
    print(f"   criada com: {comando}")

    # ── 3. Colunas e tipos ────────────────────────────────────────────────
    # PRAGMA table_info devolve: (posição, nome, tipo, not null?, valor padrão, chave primária?)
    for posicao, coluna, tipo, _nao_nulo, _padrao, chave in banco.execute(f'PRAGMA table_info("{nome}")'):
        marca = " 🔑 chave primária" if chave else ""
        print(f"   {posicao}. {coluna:<12} {tipo:<8}{marca}")

    # ── 4. As 3 primeiras linhas (LIMIT) ──────────────────────────────────
    print("   primeiras linhas:")
    for linha in banco.execute(f'SELECT * FROM "{nome}" LIMIT 3'):
        print(f"     {linha}")

# ── 5. Uma pista estrutural ───────────────────────────────────────────────
# A coluna pedidos.conta_id guarda um número que bate com contas_recebimento.id.
# Isso é uma "chave estrangeira": é o fio que liga cada pedido à conta que recebeu o PIX.
print("\n🧵 Pista: pedidos.conta_id ↔ contas_recebimento.id. No próximo script usamos esse fio")
print("   para descobrir quem recebeu cada real.")
banco.close()

### `br/02_quem_recebeu.py` — Caso Vitrine Fantasma: quem recebeu o dinheiro? (JOIN + GROUP BY)

In [ ]:
%cd -q {PASTA_NB}
"""
Caso Vitrine Fantasma: quem recebeu o dinheiro? (JOIN + GROUP BY)

Carta 29 (rank B) — Jutsu do Arquivo Morto (aplicação no caso)

O que você aprende:
  • JOIN: juntar duas tabelas pelo campo em comum (pedidos.conta_id = contas_recebimento.id);
  • GROUP BY + COUNT/SUM/MIN/MAX: resumir por conta laranja;
  • por que o REAL do SQLite tem o mesmo problema do float (e como contornar);
  • a cruzar o dump com a planilha de denúncias: as fontes concordam?

Como rodar (dentro da pasta dia-08-bancos-de-dados):
    python br/02_quem_recebeu.py
"""

import csv
import sqlite3
from decimal import Decimal
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"


def reais(valor):
    """Decimal → 'R$ 21.242,10'."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


banco = sqlite3.connect(":memory:")
banco.executescript((PASTA_CASO / "painel_dump.sql").read_text(encoding="utf-8"))

# ── 1. A consulta principal ───────────────────────────────────────────────
# Leia de baixo para cima: pegue os pedidos (p), junte com a conta (c) cujo id
# é igual ao conta_id do pedido, agrupe por conta e calcule os totais de cada grupo.
# ROUND(valor * 100) transforma reais em CENTAVOS inteiros: soma de inteiros não erra.
CONSULTA = """
    SELECT c.titular,
           c.banco,
           c.chave_pix,
           COUNT(*)                            AS pedidos,
           SUM(CAST(ROUND(p.valor * 100) AS INTEGER)) AS centavos,
           MIN(p.data)                         AS primeiro,
           MAX(p.data)                         AS ultimo
      FROM pedidos AS p
      JOIN contas_recebimento AS c ON c.id = p.conta_id
     GROUP BY c.id
     ORDER BY centavos DESC
"""

print("💸 Dinheiro recebido por conta (dump do painel):\n")
total_dump = Decimal("0")
for titular, banco_nome, chave, pedidos, centavos, primeiro, ultimo in banco.execute(CONSULTA):
    valor = Decimal(centavos) / 100                  # centavos → reais, sem erro
    total_dump += valor
    print(f"👤 {titular}  ({banco_nome})")
    print(f"   chave {chave}")
    print(f"   {pedidos} pedidos · {reais(valor)} · de {primeiro} a {ultimo}\n")
print(f"🧮 Total no dump: {reais(total_dump)}")

# ── 2. Por que não SUM(valor) direto? ─────────────────────────────────────
bruto = banco.execute("SELECT SUM(valor) FROM pedidos WHERE conta_id = 1").fetchone()[0]
print(f"\n💡 SUM(valor) direto na conta 1 dá {bruto!r}: o REAL do SQLite é float. Centavos, sempre.")

# ── 3. Conferir com a planilha das vítimas ────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))
total_planilha = sum(Decimal(d["valor"]) for d in denuncias)
print(f"\n📄 Total na planilha de denúncias: {reais(total_planilha)}")
if total_planilha == total_dump:
    print("✅ O painel da loja e as vítimas contam a MESMA história, real por real.")
else:
    print(f"🚩 Diferença de {reais(abs(total_planilha - total_dump))}: há vítimas que não denunciaram (ou vice-versa).")

# ── 4. Pergunta de investigador: algum titular aparece nos dois lados? ────
# O vendedor que assina "Kau" aparece nos registros como Kauã Teixeira Brandão (caso/README.md).
suspeitas = banco.execute(
    "SELECT titular FROM contas_recebimento WHERE titular LIKE ?", ("%Brandão%",)).fetchall()
for (titular,) in suspeitas:
    print(f"\n🔎 '{titular}' tem o mesmo sobrenome do vendedor 'Kau'. Coincidência ou empresa de fachada?")
    print("   Anote como hipótese, não como conclusão: sobrenome igual não prova vínculo.")
banco.close()

### `br/03_banco_do_caso.py` — Criar o banco do caso (saidas/caso.db) a partir da planilha e consultar com segurança.

In [ ]:
%cd -q {PASTA_NB}
"""
Criar o banco do caso (saidas/caso.db) a partir da planilha e consultar com segurança.

Carta 29 (rank B) — Jutsu do Arquivo Morto (parte 3)

O que você aprende:
  • a criar um arquivo SQLite de verdade e importar um CSV com executemany;
  • a guardar dinheiro em CENTAVOS (INTEGER) para nunca errar soma;
  • consultas parametrizadas com "?" e por que isso evita SQL injection;
  • consultas por cidade e por mês com GROUP BY e strftime.

Como rodar (dentro da pasta dia-08-bancos-de-dados):
    python br/03_banco_do_caso.py
"""

import csv
import sqlite3
from decimal import Decimal
from pathlib import Path

CIDADE = "Belo Horizonte"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"


def reais(centavos):
    valor = Decimal(centavos) / 100
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


# ── 1. Criar o arquivo do banco (do zero, a cada execução) ────────────────
SAIDAS.mkdir(exist_ok=True)
caminho_db = SAIDAS / "caso.db"
caminho_db.unlink(missing_ok=True)            # apaga a versão anterior, se houver
banco = sqlite3.connect(caminho_db)

banco.execute("""
    CREATE TABLE denuncias (
        id             TEXT PRIMARY KEY,
        data           TEXT NOT NULL,      -- AAAA-MM-DD (texto ISO ordena certinho)
        cidade         TEXT,
        uf             TEXT,
        valor_centavos INTEGER NOT NULL,   -- 149.90 vira 14990
        produto        TEXT,
        chave_pix      TEXT
    )
""")

# ── 2. Importar o CSV ─────────────────────────────────────────────────────
# Minimização (LGPD): NÃO levamos nome, CPF nem telefone para este banco de análise.
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as arquivo:
    linhas = [
        (d["id"], d["data"], d["cidade"], d["uf"], int(Decimal(d["valor"]) * 100), d["produto"],
         d["chave_pix_destino"])
        for d in csv.DictReader(arquivo, delimiter=";")
    ]
# executemany roda o mesmo INSERT para cada tupla. Os "?" são preenchidos pelo SQLite.
banco.executemany("INSERT INTO denuncias VALUES (?, ?, ?, ?, ?, ?, ?)", linhas)
banco.commit()                                 # grava de verdade no arquivo
print(f"🗄️  {len(linhas)} denúncias importadas para {caminho_db.relative_to(PASTA_DIA)}\n")

# ── 3. Consulta por mês ───────────────────────────────────────────────────
print("📅 Por mês:")
for mes, qtd, centavos in banco.execute("""
        SELECT strftime('%Y-%m', data) AS mes, COUNT(*), SUM(valor_centavos)
          FROM denuncias GROUP BY mes ORDER BY mes"""):
    print(f"   {mes}  {qtd:>2} denúncias  {reais(centavos):>13}")

# ── 4. Consulta por cidade (como foi digitada; a padronização foi o Dia 7) ─
print("\n🏙️  Top 5 cidades (grafia original):")
for cidade, qtd in banco.execute(
        "SELECT cidade, COUNT(*) AS n FROM denuncias GROUP BY cidade ORDER BY n DESC, cidade LIMIT 5"):
    print(f"   {cidade:<22} {qtd}")

# ── 5. Consulta PARAMETRIZADA ─────────────────────────────────────────────
# O valor do usuário vai SEPARADO do comando, no lugar do "?". O SQLite trata como dado, nunca
# como código. lower() dos dois lados faz "belo horizonte" e "Belo Horizonte" baterem.
consulta_segura = "SELECT COUNT(*), SUM(valor_centavos) FROM denuncias WHERE lower(cidade) = lower(?)"
qtd, centavos = banco.execute(consulta_segura, (CIDADE,)).fetchone()
print(f"\n🔎 Denúncias em {CIDADE!r}: {qtd} · {reais(centavos or 0)}")
print("   (\"Belo Horizone\", com erro de digitação, NÃO entra: por isso padronizamos no Dia 7.)")

# ── 6. Por que NUNCA montar SQL juntando texto ────────────────────────────
ataque = "x' OR '1'='1"                         # o que um usuário malicioso digitaria
perigosa = f"SELECT COUNT(*) FROM denuncias WHERE cidade = '{ataque}'"
print("\n☠️  SQL injection, demonstração:")
print(f"   comando montado com f-string: {perigosa}")
print(f"   resultado: {banco.execute(perigosa).fetchone()[0]} linhas  ← devolveu TUDO!")
segura = banco.execute("SELECT COUNT(*) FROM denuncias WHERE cidade = ?", (ataque,)).fetchone()[0]
print(f"   mesma entrada com '?': {segura} linhas  ← tratado como texto comum, inofensivo")
banco.close()

## ⚖️ Ética, LGPD e OPSEC

- **Vazamento real não é material de estudo.** Baixar, guardar ou compartilhar dumps com dados
  pessoais de terceiros viola a LGPD e pode configurar crime. Invadir um sistema para obtê-los é o
  crime do **art. 154-A do Código Penal** (invasão de dispositivo informático, Lei 12.737/2012, com
  penas aumentadas pela Lei 14.155/2021). Mesmo um arquivo "achado" no Google não vira dado público.
- **Dorks do GHDB** servem para entender **exposição**: avaliar se uma organização que você está
  autorizado a proteger deixou arquivo aberto. Achou algo de terceiros? Não abra; registre a URL e
  avise o responsável ou a autoridade competente.
- **Prova recebida exige cadeia de custódia.** Hash SHA-256 antes de abrir, registro de quem entregou,
  quando e como (arts. 158-A a 158-F do CPP; o Dia 14 aprofunda). Trabalhe numa cópia; o original
  fica lacrado.
- **Minimize.** O banco de análise (`caso.db`) não leva nome, CPF nem telefone das vítimas, porque as
  perguntas do dia não precisam deles.
- **Não use conversores online** para arquivos de caso. Tudo o que você precisa está no `sqlite3`.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Para que serve a tabela especial sqlite_master?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Guarda as senhas do banco", "B) Lista as tabelas (e outros objetos) e o comando que criou cada uma", "C) Faz backup automático", "D) Mostra quem acessou o banco"]
correta = "B) Lista as tabelas (e outros objetos) e o comando que criou cada uma"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "É o índice do banco SQLite. Consultar sqlite_master é o primeiro passo para conhecer um dump desconhecido.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Por que usar banco.execute('... WHERE cidade = ?', (cidade,)) em vez de uma f-string?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque é mais curto", "B) Porque o '?' separa o dado do comando e impede SQL injection", "C) Porque f-string não funciona com SQLite", "D) Porque o '?' deixa a consulta mais rápida na internet"]
correta = "B) Porque o '?' separa o dado do comando e impede SQL injection"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Com '?', o valor nunca é interpretado como código SQL. Com f-string, uma entrada como x' OR '1'='1 muda o sentido do comando.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Um colega encontrou, por um dork, um .sql exposto com e-mails e CPFs de clientes de uma loja real. O que fazer?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Baixar e guardar para usar depois", "B) Postar o link nas redes para alertar", "C) Não baixar nem abrir; registrar a URL e comunicar o responsável/autoridade pelos canais adequados", "D) Converter para CSV e compartilhar com a equipe"]
correta = "C) Não baixar nem abrir; registrar a URL e comunicar o responsável/autoridade pelos canais adequados"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Mexer em vazamento real expõe as vítimas e pode ser crime. O caminho é registrar, não manipular, e notificar quem pode corrigir.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 8: Conciliação painel × denúncias

**Carta 50 · Rank S**

## A missão

O dump do painel diz quem recebeu. A planilha diz quem denunciou. Um advogado de defesa vai perguntar:
*"como vocês sabem que o pedido 14 do painel é a mesma pessoa da denúncia DEN-014?"*. A Iara quer a
resposta pronta, linha por linha.

Escreva um script que:

1. Carregue `caso/dados/painel_dump.sql` e importe `caso/dados/denuncias.csv` **no mesmo banco**
   SQLite (sem levar CPF nem telefone, que não são necessários para conciliar).
2. Ligue cada denúncia ao pedido do painel por **nome + data + valor** (em centavos), com
   `LEFT JOIN`, para que nenhuma denúncia suma do resultado.
3. Marque 🚩 quando: a denúncia não tiver pedido correspondente, ou a chave PIX que a vítima informou
   for diferente da chave da conta que o painel registrou.
4. Conte os **pedidos sem denúncia** (vítimas que ainda não procuraram a polícia).
5. Mostre a **linha do tempo** das contas laranja: primeira e última data de uso, pedidos e total.
6. Grave `saidas/conciliacao.csv` (`;`, `utf-8-sig`).
7. Use só consultas com `?` para qualquer valor que venha de fora.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Soma em centavos inteiros (nada de `SUM(valor)` em float)
- [ ] As 37 denúncias aparecem no CSV, com a situação de cada uma
- [ ] A linha do tempo mostra em que ordem as três contas entraram em uso

## Dica

`NOT EXISTS (SELECT 1 FROM … WHERE …)` responde "quais pedidos não têm denúncia?" sem precisar de laço
em Python.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 8 — Conciliação: painel da loja × denúncias.

Coloca o dump do painel e a planilha de denúncias no MESMO banco SQLite, liga
cada denúncia ao pedido correspondente (cliente + data + valor), confere se a
chave PIX informada pela vítima é a da conta que o painel registrou, aponta
divergências e monta a linha do tempo de entrada em uso de cada conta laranja.
Grava saidas/conciliacao.csv. 100% offline (dados fictícios de caso/dados/).
"""

import csv
import sqlite3
from decimal import Decimal
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"


def reais(centavos):
    return "R$ " + f"{Decimal(centavos) / 100:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


banco = sqlite3.connect(":memory:")
banco.executescript((PASTA_CASO / "painel_dump.sql").read_text(encoding="utf-8"))

# Tabela com as denúncias (sem CPF e telefone: não precisamos deles para conciliar)
banco.execute("""CREATE TABLE denuncias (id TEXT PRIMARY KEY, data TEXT, nome TEXT,
                 valor_centavos INTEGER, chave_pix TEXT)""")
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as f:
    banco.executemany("INSERT INTO denuncias VALUES (?, ?, ?, ?, ?)", [
        (d["id"], d["data"], d["nome_vitima"], int(Decimal(d["valor"]) * 100), d["chave_pix_destino"])
        for d in csv.DictReader(f, delimiter=";")])

# LEFT JOIN: toda denúncia aparece, mesmo sem pedido correspondente (aí os campos do pedido vêm NULL).
CONCILIA = """
SELECT d.id, d.data, d.nome, d.valor_centavos, d.chave_pix,
       p.id AS pedido, c.titular, c.chave_pix AS chave_painel
  FROM denuncias AS d
  LEFT JOIN pedidos AS p
         ON p.cliente = d.nome AND p.data = d.data
        AND CAST(ROUND(p.valor * 100) AS INTEGER) = d.valor_centavos
  LEFT JOIN contas_recebimento AS c ON c.id = p.conta_id
 ORDER BY d.id
"""
linhas, problemas = [], 0
for den, data, nome, centavos, chave_vitima, pedido, titular, chave_painel in banco.execute(CONCILIA):
    if pedido is None:
        situacao = "🚩 denúncia sem pedido no painel"
    elif chave_vitima != chave_painel:
        situacao = "🚩 chave PIX diverge"
    else:
        situacao = "ok"
    problemas += situacao != "ok"
    linhas.append([den, data, pedido or "", titular or "", reais(centavos), situacao])

# Pedidos que ninguém denunciou ("vítimas silenciosas")
silenciosos = banco.execute("""
    SELECT COUNT(*) FROM pedidos AS p
     WHERE NOT EXISTS (SELECT 1 FROM denuncias AS d
                        WHERE d.nome = p.cliente AND d.data = p.data
                          AND d.valor_centavos = CAST(ROUND(p.valor * 100) AS INTEGER))""").fetchone()[0]

print(f"🔗 {len(linhas)} denúncias conferidas · {problemas} divergência(s) · "
      f"{silenciosos} pedido(s) sem denúncia (vítimas silenciosas)")

# Linha do tempo: quando cada conta começou a receber? (rotação de chaves)
print("\n🕰️  Entrada em uso de cada conta laranja:")
for titular, inicio, fim, n, centavos in banco.execute("""
        SELECT c.titular, MIN(p.data), MAX(p.data), COUNT(*), SUM(CAST(ROUND(p.valor*100) AS INTEGER))
          FROM pedidos p JOIN contas_recebimento c ON c.id = p.conta_id
         GROUP BY c.id ORDER BY MIN(p.data)"""):
    print(f"   {inicio} → {fim}  {titular:<28} {n:>2} pedidos  {reais(centavos)}")
print("   💡 Uma conta nova entra a cada poucos dias e todas seguem ativas até o fim: hipótese de")
print("      espalhar o dinheiro para não chamar atenção de um banco só. Hipótese, não conclusão.")

SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "conciliacao.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["denuncia", "data", "pedido_painel", "titular_recebedor", "valor", "situacao"])
    w.writerows(linhas)
print("\n💾 saidas/conciliacao.csv gravado.")
banco.close()

## 🔎 Inovação OSINT do dia

**datasette.** Ferramenta open source criada por Simon Willison que transforma qualquer arquivo SQLite
num site navegável: busca, filtros, facetas, exportação para CSV/JSON e um campo para escrever SQL.
Com `pip install datasette` e `datasette saidas/caso.db`, a equipe inteira explora o banco do caso
pelo navegador, sem programar. Rode sempre **localmente** (`127.0.0.1`) para dados sensíveis.
Para quem quer ficar na stdlib: `sqlite3` + `csv` resolvem tudo deste dia.

## 📚 Para ir além

- Módulo `sqlite3` do Python (em português): <https://docs.python.org/pt-br/3/library/sqlite3.html>
- Funções de data do SQLite (`strftime`): <https://www.sqlite.org/lang_datefunc.html>
- Google Hacking Database (Exploit-DB): <https://www.exploit-db.com/google-hacking-database>
- datasette: <https://datasette.io/>
- OWASP sobre SQL injection: <https://owasp.org/www-community/attacks/SQL_Injection>

## ✅ Checklist

- [ ] Li o resumo do Dia 8 original e rodei a reconstrução (dump → CSV)
- [ ] Calculei o hash do dump antes de abrir
- [ ] Listei tabelas e colunas de um banco desconhecido
- [ ] Descobri quanto cada conta laranja recebeu, em centavos
- [ ] Vi o SQL injection acontecer (e ser bloqueado com `?`)
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 8](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)